# Stage 4 — Small Reasoning Probe: Molmo2 vs Qwen3-VL vs InternVL3

Self-contained (no `git clone`/`src` import).

**What this tests:** not symbol detection — a short counting/comparison question over
the image ("which symbol type is most common here, and roughly how many?"), checked
against real ground-truth counts from the Kaggle labels. This is a small, indicative
signal about general reasoning/instruction-following, not a rigorous benchmark — one
prompt, a handful of tiles, judged by eye against the real answer.

**Why this test specifically:** Molmo2 is trained heavily around pointing-style output;
Qwen3-VL/InternVL3 are general-purpose VLMs more used to open-ended Q&A. If Molmo2
struggles to give a coherent counting/comparison answer while the other two manage it,
that's a real (if small) data point in the direction of the "unproven on reasoning"
caveat already in CLAUDE.md — not proof, an indication.

One model is loaded, tested, and freed before the next loads, to fit on one GPU.

## 1. Check GPU, install Kaggle CLI (no Google Drive needed)

Skips `drive.mount()` entirely — that's what's been triggering the account
verification-code friction. Downloads a handful of P&ID images directly from Kaggle's
own API into this runtime's local storage instead.

In [3]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q kaggle

NVIDIA A100-SXM4-80GB, 81920 MiB, 30038 MiB


## 1b. Kaggle auth — paste your token here (not stored on Drive)

Get a token from kaggle.com → Settings → API → Create New Token, then paste just the
token string below. This is a Kaggle account credential, unrelated to the Google/Drive
auth that's been causing friction.

In [5]:
import os

KAGGLE_TOKEN = "KGAT_f7b679c152481eefca6f74c03bf4ec04"  # <-- edit this

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
token_path = os.path.expanduser("~/.kaggle/access_token")
with open(token_path, "w") as f:
    f.write(KAGGLE_TOKEN)
os.chmod(token_path, 0o600)
print("Kaggle token installed for this session.")

Kaggle token installed for this session.


## 1c. Download a small sample of the P&ID dataset (not the full 30k images)

In [11]:
KAGGLE_LOCAL_DIR = "/content/kaggle_pid_symbols"

!rm -rf /content/kaggle_dl {KAGGLE_LOCAL_DIR}
!mkdir -p {KAGGLE_LOCAL_DIR}
!kaggle datasets download -d hristohristov21/pid-symbols -p {KAGGLE_LOCAL_DIR} --unzip

# The zip itself contains folders literally named "images (3)", "labels (2)" etc (same
# quirk we hit earlier this session fixing the original broken Drive copy) - normalize
# to plain names.
import os
from pathlib import Path
_kdir = Path(KAGGLE_LOCAL_DIR)
_renames = {
    "images (3)": "images", "labels (2)": "labels",
    "train (2).txt": "train.txt", "val (1).txt": "val.txt",
}
for old_name, new_name in _renames.items():
    old_path = _kdir / old_name
    if old_path.exists():
        old_path.rename(_kdir / new_name)
        print(f"Renamed: {old_name} -> {new_name}")

print("Downloaded to", KAGGLE_LOCAL_DIR)
print("Contents:", [p.name for p in _kdir.iterdir()])


Dataset URL: https://www.kaggle.com/datasets/hristohristov21/pid-symbols
License(s): other
100% 1.41G/1.41G [01:09<00:00, 21.7MB/s]

Renamed: images (3) -> images
Renamed: labels (2) -> labels
Renamed: train (2).txt -> train.txt
Renamed: val (1).txt -> val.txt
Downloaded to /content/kaggle_pid_symbols
Contents: ['val.txt', 'labels', 'images', 'train.txt']


## 2. Pick 3 labeled Kaggle tiles + compute real ground-truth majority class

(No model loaded yet — just picking tiles and reading their real labels, so we know
the correct answer before asking anything.)

In [9]:
import json, random
from collections import Counter
from pathlib import Path
from PIL import Image

kaggle_p = Path(KAGGLE_LOCAL_DIR)

# No classes.json available without Drive — using the same names documented in
# Stage4_Phase2_Data_Preparation.ipynb (class 0 confirmed unused, verified against
# real data: class 0 has 0 instances, class 21 has ~3x every other class).
class_names = {
    "0": "Not_used", "1": "Gate_Valve", "2": "Ball_Valve", "3": "Globe_valve_NO",
    "4": "Gate_valve_NO", "5": "Globe_valve_NO", "6": "Butterfly_valve", "7": "Plug_valve",
    "8": "Check_valve", "9": "Diaphragm_valve", "10": "Needle_valve",
    "11": "Half_Filled_Gate_Valve", "12": "Gate_Valve_NC", "13": "Globle_valve_NC",
    "14": "Control_Valve", "15": "Rotary_Valve", "16": "Ball_valve_NC", "17": "Paddle_blind",
    "18": "Spectacle_blind_Closed", "19": "Spectacle_blind_Open", "20": "Reducer",
    "21": "Flange_or_Nozzle", "22": "Rupture_disk", "23": "Pipe_Insulation_or_Tracing",
    "24": "Flow_Arrow", "25": "sight_glass", "26": "Instrument_Field", "27": "Instrument_Field",
    "28": "Instrument_Panel", "29": "Instrument_Aux_Panel", "30": "box",
    "31": "Instrument_Panel", "32": "box",
}

random.seed(1)
candidates = []
for lbl in (kaggle_p / "labels").glob("*.txt"):
    lines = [l for l in lbl.read_text().splitlines() if l.strip()]
    if len(lines) < 6:
        continue
    counts = Counter(l.split()[0] for l in lines)
    top_cls, top_n = counts.most_common(1)[0]
    # only keep tiles with an unambiguous majority (not a 3-way tie etc.)
    if top_n >= 3 and (len(counts) == 1 or top_n > counts.most_common(2)[1][1]):
        candidates.append((lbl, top_cls, top_n, len(lines)))
    if len(candidates) >= 200:
        break

probe_tiles = random.sample(candidates, 3)
for lbl, top_cls, top_n, total in probe_tiles:
    name = class_names.get(top_cls, f"class_{top_cls}")
    print(f"{lbl.stem:20s} majority={name:20s} count={top_n}  (total boxes in tile: {total})")

NameError: name 'KAGGLE_LOCAL_DIR' is not defined

In [10]:
KAGGLE_LOCAL_DIR = "/content/kaggle_pid_symbols"

In [13]:
print("kaggle_p:", kaggle_p, "exists:", kaggle_p.exists())
print("images dir exists:", (kaggle_p / "images").exists())
print("labels dir exists:", (kaggle_p / "labels").exists())
if (kaggle_p / "labels").exists():
    label_files = list((kaggle_p / "labels").glob("*.txt"))
    print("label files found:", len(label_files))
print("candidates found:", len(candidates))

!ls -la /content/kaggle_pid_symbols/

kaggle_p: /content/kaggle_pid_symbols exists: True
images dir exists: True
labels dir exists: True
label files found: 30000
candidates found: 200
total 2856
drwxr-xr-x 4 root root    4096 Jul 10 14:59 .
drwxr-xr-x 1 root root    4096 Jul 10 14:57 ..
drwxr-xr-x 2 root root 1064960 Jul 10 14:59 images
drwxr-xr-x 2 root root 1069056 Jul 10 14:59 labels
-rw-r--r-- 1 root root  694345 Jul 10 14:59 train.txt
-rw-r--r-- 1 root root   77104 Jul 10 14:59 val.txt


In [7]:
!ls -la /content/kaggle_dl/ 2>&1 | head -20
!ls -la /content/kaggle_pid_symbols/ 2>&1 | head -20

total 8
drwxr-xr-x 2 root root 4096 Jul 10 14:50 .
drwxr-xr-x 1 root root 4096 Jul 10 14:50 ..
total 2848
drwxr-xr-x 4 root root    4096 Jul 10 14:50 .
drwxr-xr-x 1 root root    4096 Jul 10 14:50 ..
drwxr-xr-x 2 root root 1064960 Jul 10 14:50 images (3)
drwxr-xr-x 2 root root 1069056 Jul 10 14:50 labels (2)
-rw-r--r-- 1 root root  694345 Jul 10 14:50 train (2).txt
-rw-r--r-- 1 root root   77104 Jul 10 14:50 val (1).txt


## 3. The reasoning prompt (same question, all 3 candidates)

In [8]:
REASONING_PROMPT = (
    "Look at this P&ID tile. Which type of symbol appears most frequently in this image, "
    "and approximately how many of that symbol are there? Answer in one or two sentences."
)

def show_probe_images():
    for lbl, top_cls, top_n, total in probe_tiles:
        img_path = kaggle_p / "images" / f"{lbl.stem}.jpg"
        print(f"--- {lbl.stem} --- ground truth: {class_names.get(top_cls, top_cls)} x{top_n} (of {total} total boxes)")
        display(Image.open(img_path))

## 4. Qwen3-VL

In [15]:
!pip install -q transformers accelerate qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 80.9 MB/s eta 0:00:00:00:0100:01


In [16]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")
qwen_model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct", dtype=torch.bfloat16, device_map="cuda"
)
print("Qwen3-VL loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

for lbl, top_cls, top_n, total in probe_tiles:
    img = Image.open(kaggle_p / "images" / f"{lbl.stem}.jpg").convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": REASONING_PROMPT}]}]
    inputs = qwen_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    ).to(qwen_model.device)
    out = qwen_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = qwen_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    print(f"\n=== {lbl.stem} (ground truth: {class_names.get(top_cls, top_cls)} x{top_n}) ===")
    print("Qwen3-VL:", answer.strip())

del qwen_model, qwen_processor
torch.cuda.empty_cache()
print("\nFreed Qwen3-VL from GPU.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Qwen3-VL loaded. VRAM: 17.5 GB

=== 109_3200_1920 (ground truth: Flange_or_Nozzle x4) ===
Qwen3-VL: The most frequent symbol is the “STA” (station) symbol, appearing twice in the image. There are also several other symbols like valves, flow meters, and pressure gauges, but “STA” is the most repeated.

=== 235_0_4480 (ground truth: Flange_or_Nozzle x4) ===
Qwen3-VL: The most frequent symbol is the valve (represented by the cross-hatched diamond shape), appearing approximately 4 times in the image.

=== 136_2560_2560 (ground truth: box x3) ===
Qwen3-VL: The “STA” symbol (a square with “STA” inside) appears most frequently, and there are approximately 4 of them visible in the image.

Freed Qwen3-VL from GPU.


## 5. InternVL3

In [17]:
internvl_processor = AutoProcessor.from_pretrained("OpenGVLab/InternVL3-8B-hf")
internvl_model = AutoModelForImageTextToText.from_pretrained(
    "OpenGVLab/InternVL3-8B-hf", dtype=torch.bfloat16, device_map="cuda"
)
print("InternVL3 loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

for lbl, top_cls, top_n, total in probe_tiles:
    img = Image.open(kaggle_p / "images" / f"{lbl.stem}.jpg").convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": REASONING_PROMPT}]}]
    inputs = internvl_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    ).to(internvl_model.device)
    out = internvl_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = internvl_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    print(f"\n=== {lbl.stem} (ground truth: {class_names.get(top_cls, top_cls)} x{top_n}) ===")
    print("InternVL3:", answer.strip())

del internvl_model, internvl_processor
torch.cuda.empty_cache()
print("\nFreed InternVL3 from GPU.")

processor_config.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/481 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/6.86k [00:00<?, ?B/s]

...

InternVL3 loaded. VRAM: 15.9 GB


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



=== 109_3200_1920 (ground truth: Flange_or_Nozzle x4) ===
InternVL3: The most frequent symbol is the rectangle, and there are approximately 10 of them.


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



=== 235_0_4480 (ground truth: Flange_or_Nozzle x4) ===
InternVL3: The most frequent symbol is the circle with a line through it, representing a valve. There are approximately 10 of these symbols.

=== 136_2560_2560 (ground truth: box x3) ===
InternVL3: The triangle symbol appears most frequently, and there are approximately six of them.

Freed InternVL3 from GPU.


**Prereqs for standalone run — re-picks tiles + prompt (needed again after any runtime restart)**

In [12]:
KAGGLE_LOCAL_DIR = "/content/kaggle_pid_symbols"

In [13]:
import json, random
from collections import Counter
from pathlib import Path
from PIL import Image

kaggle_p = Path(KAGGLE_LOCAL_DIR)

# No classes.json available without Drive — using the same names documented in
# Stage4_Phase2_Data_Preparation.ipynb (class 0 confirmed unused, verified against
# real data: class 0 has 0 instances, class 21 has ~3x every other class).
class_names = {
    "0": "Not_used", "1": "Gate_Valve", "2": "Ball_Valve", "3": "Globe_valve_NO",
    "4": "Gate_valve_NO", "5": "Globe_valve_NO", "6": "Butterfly_valve", "7": "Plug_valve",
    "8": "Check_valve", "9": "Diaphragm_valve", "10": "Needle_valve",
    "11": "Half_Filled_Gate_Valve", "12": "Gate_Valve_NC", "13": "Globle_valve_NC",
    "14": "Control_Valve", "15": "Rotary_Valve", "16": "Ball_valve_NC", "17": "Paddle_blind",
    "18": "Spectacle_blind_Closed", "19": "Spectacle_blind_Open", "20": "Reducer",
    "21": "Flange_or_Nozzle", "22": "Rupture_disk", "23": "Pipe_Insulation_or_Tracing",
    "24": "Flow_Arrow", "25": "sight_glass", "26": "Instrument_Field", "27": "Instrument_Field",
    "28": "Instrument_Panel", "29": "Instrument_Aux_Panel", "30": "box",
    "31": "Instrument_Panel", "32": "box",
}

random.seed(1)
candidates = []
for lbl in (kaggle_p / "labels").glob("*.txt"):
    lines = [l for l in lbl.read_text().splitlines() if l.strip()]
    if len(lines) < 6:
        continue
    counts = Counter(l.split()[0] for l in lines)
    top_cls, top_n = counts.most_common(1)[0]
    # only keep tiles with an unambiguous majority (not a 3-way tie etc.)
    if top_n >= 3 and (len(counts) == 1 or top_n > counts.most_common(2)[1][1]):
        candidates.append((lbl, top_cls, top_n, len(lines)))
    if len(candidates) >= 200:
        break

probe_tiles = random.sample(candidates, 3)
for lbl, top_cls, top_n, total in probe_tiles:
    name = class_names.get(top_cls, f"class_{top_cls}")
    print(f"{lbl.stem:20s} majority={name:20s} count={top_n}  (total boxes in tile: {total})")

109_3200_1920        majority=Flange_or_Nozzle     count=4  (total boxes in tile: 14)
235_0_4480           majority=Flange_or_Nozzle     count=4  (total boxes in tile: 11)
136_2560_2560        majority=box                  count=3  (total boxes in tile: 12)


In [14]:
REASONING_PROMPT = (
    "Look at this P&ID tile. Which type of symbol appears most frequently in this image, "
    "and approximately how many of that symbol are there? Answer in one or two sentences."
)

def show_probe_images():
    for lbl, top_cls, top_n, total in probe_tiles:
        img_path = kaggle_p / "images" / f"{lbl.stem}.jpg"
        print(f"--- {lbl.stem} --- ground truth: {class_names.get(top_cls, top_cls)} x{top_n} (of {total} total boxes)")
        display(Image.open(img_path))

## 6. Molmo2-O-7B

Requires `transformers==4.57.1` — **restart the runtime after this install cell**, then
re-run cells 2-3 (tile selection) before continuing to the load+probe cell below.

In [15]:
!pip install -q transformers==4.57.1 accelerate

In [16]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

molmo_processor = AutoProcessor.from_pretrained("allenai/Molmo2-O-7B", trust_remote_code=True, dtype="auto")
molmo_model = AutoModelForImageTextToText.from_pretrained(
    "allenai/Molmo2-O-7B", trust_remote_code=True, dtype="auto", device_map="cuda"
)
print("Molmo2 loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

for lbl, top_cls, top_n, total in probe_tiles:
    img = Image.open(kaggle_p / "images" / f"{lbl.stem}.jpg").convert("RGB")
    messages = [{"role": "user", "content": [{"type": "text", "text": REASONING_PROMPT}, {"type": "image", "image": img}]}]
    inputs = molmo_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    ).to(molmo_model.device)
    out = molmo_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = molmo_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    print(f"\n=== {lbl.stem} (ground truth: {class_names.get(top_cls, top_cls)} x{top_n}) ===")
    print("Molmo2:", answer.strip())

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Molmo2 loaded. VRAM: 31.0 GB

=== 109_3200_1920 (ground truth: Flange_or_Nozzle x4) ===
Molmo2: The most frequent symbol in this P&ID tile is the circle with a line through it, which appears approximately 10 times. These symbols are scattered throughout the image, representing various components or connections in the process diagram.

=== 235_0_4480 (ground truth: Flange_or_Nozzle x4) ===
Molmo2: The most frequent symbol in this P&ID tile is the valve symbol, which appears approximately 6 times. These valve symbols are represented by the hourglass shape with two triangles pointing towards each other, and they are scattered throughout the image, connecting various lines and components of the process diagram.

=== 136_2560_2560 (ground truth: box x3) ===
Molmo2: The most frequent symbol in this P&ID tile image is the square box with the letters "STA" inside. There are approximately 5 of these symbols visible in the image.


## 7. What to look for

For each tile, compare the three answers against the printed ground truth:
- Does the answer name a plausible symbol type, or is it garbled/off-topic/just points?
- Is the count in the right ballpark (doesn't need to be exact)?
- Does it actually answer the comparison question, or does it just describe the image generically?

Paste the three candidates' raw answers back — this determines what (if anything) goes
into the write-up. A model doing noticeably worse here isn't proof of anything on its own,
but it's a real, checkable data point, not an assumption.